In [14]:
##### importing custom modules from the projects folder
import sys
from pathlib import Path
# Start at current working directory
current = Path.cwd()
# Walk up the tree until config.py is found or root is reached
for parent in [current] + list(current.parents):
    config_path = parent / "config.py"
    if config_path.exists():
        sys.path.append(str(parent))
        import config # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
        break
else:
    raise FileNotFoundError("config.py not found in any parent directories")

import scripts.scrapers.nbaScraper as ns
import scripts.scrapers.actNetApi as ans
import scripts.scrapers.nbApi as nbapi

from datetime import datetime, timedelta
# -----------------------------------------------
# -------------------------------- PARAMS
leagues =  None  # None or list ['nba', 'nhl', 'nfl', 'mlb', 'wnba'] # NONE looks for all sports
specified =  []  ####  [specific prop] or [] for all props, *****only works with a single league in leagues

# day adjustment from today (date of running script), negative = dates into the past
dayJump = 0
# date can be a list of dates if multiple need scraping 'YYYY-MM-DD'
# default is to only pull today or today + dayJump
dates = [(datetime.today() + timedelta(days=dayJump)).strftime('%Y-%m-%d')]
#dates = ['2025-10-26']#,  '2025-10-02']#, '2025-09-13']

database_export = True  # add all scrapes to database
store_locally = True    # add all scrapes to class variables
season_int = 2026 # int will be the final year of the schedule season
season_str = '2025-26'
season_start_date = '2025-10-21' #'MM/DD/YYY'
season_type = 'Regular+Season' # ['Regular+Season', 'PlayIn', 'Playoffs']
per_mode = 'Totals' #['Totals', 'PerGame']
# ------------------------------------------------


In [15]:
# nba website and basketball referenece scrapers
scraper = ns.scraper(
    browser_path = str(config.BROWSER_DIR) + '\\geckodriver.exe',
    database_export = database_export, 
    store_locally = store_locally,
    pymysql_conn_str =  None
)
# assigns the date of the last time code executed as today
today = scraper.meta_data['today_dt']

# looks up the actual date for the last regular season game date. this will be used to grab the data for the completed games on the date
nbaApi = nbapi.nbaApi(
    browser_path = str(config.BROWSER_DIR) + '\\geckodriver.exe',
    database_export = database_export, 
    store_locally = store_locally,
    pymysql_conn_str =  None
)
a = nbaApi.get_last_game_date(season = season_str)
run_date = nbaApi.last_game_date
dateRange = [
    run_date, run_date
]

#prop_scraper = ans.actNetScraper(
prop_scraper = ans.actNetApi(    
    browser_path = str(config.BROWSER_DIR) + '\\geckodriver.exe',
    dates = dates,
    leagues = leagues,
    database_export = database_export, 
    store_locally = store_locally,
    second_run = False
)

# turn to False if issues loading new players
#prop_scraper.update_players = False

# update the league list to only ones with games today
#leagues = prop_scraper.check_for_league_games(date_check = None, update_class_leagues_var = True)
print('scraping for', prop_scraper.leagues, 'on', prop_scraper.dates)

Last day with games played: 2025-11-03
scraping for ['nba', 'nhl'] on ['2025-11-04']


In [16]:
# SCRAPE PROPS
print(today, 'run date...\n for', dates, 'and', prop_scraper.leagues)
############## these 2 lines were used w. old selenium setup in actNetScraper.py
#prop_scraper.scrape(sleep_secs = 3, specific_props = specified,leagues_override = prop_scraper.leagues,an_state_code = 'BC')
#prop_scraper.processScrapes(remove_dups = True,specific_props = specified)
###############
prop_scraper.scrape(
    sleep_secs = 3, 
    specific_props = specified,
    leagues_override = prop_scraper.leagues,
    an_state_code = 'BC'
)
print('html saved...\n')

prop_scraper.processScrapes(
    remove_dups = True,
    specific_props = specified
) 

if prop_scraper.scrape_error_flag:
    print(prop_scraper.scrape_errors)
    #prop_scraper.tryMissingProps()


2025-11-04 run date...
 for ['2025-11-04'] and ['nba', 'nhl']
html saved...

processing nba ...
original rows:  (1003, 20)
after dups removed:  (1003, 20)
['Jeremiah Fears' 'Collin Murray-Boyles' 'Yves Missi' 'Jared McCain'
 'Ryan Dunn']
nba odds data loaded...
prop          ast  blk  pa  pr  pra  pts  ra  reb  sb  stl  threes
propId count   94   90  91  94   94   95  91   95  84   91      84
processing nhl ...
original rows:  (1829, 20)
after dups removed:  (1829, 20)
['Tristen Nielsen' 'Taylor Makar']
nhl odds data loaded...
prop          ast  ats  gs  gs1st  gs2plus  gs3plus  gsLast  pts  sog
propId count  160  368  13    368      162       84     368  161  145


In [17]:
# SCRAPE BREF TEAM MISC
print(today, 'run date...\n')
#teams = ['GSW','DEN','POR','SAC','TOR','DAL','PHO','CHI','LAL','HOU','MIA','MEM','DET','MIL','NOP','MIN','CLE','OKC','LAC','BRK','SAS','NYK','WAS','CHO','UTA','IND','BOS','PHI','ATL','ORL']
scraper.get_bref_pos_estimates(
        base_url = 'https://www.basketball-reference.com/teams/{team}/{season}.html#pbp', 
        today_date = today,
        season = season_int,
        database_table = 'brefmisc',
        team_overrides = None
)
### CAN DELETE AFTER CONFIRMING ERROR RETRIES IN FUNCTION WORK AS EXPECTED
missing_teams = scraper.scrape_errors['brefmisc']['url']
if len(missing_teams):
        print('missing:', missing_teams)
        for i in missing_teams:
                scraper.get_bref_pos_estimates(
                        base_url = 'https://www.basketball-reference.com/teams/{team}/{season}.html#pbp', 
                        today_date = today,
                        season = season_int,
                       database_table = 'brefmisc',
                        team_overrides = [i[0]]
               )


2025-11-04 run date...

bref player pos estimates scraped, 30 teams...


In [13]:
# E SP N scrape for the days game odds
# ---------------------------------------------------------------- #
scraper.get_game_odds(database_table = 'gameOdds')


In [18]:
# NBA API hits
# ---------------------------------------------------------------- #
# --- TEAM --- #
print(today, 'run date...\n')
nbaApi.request_nba_team_shotzone_data(
    season = season_str, #'YYYY-YY'
    start_date = run_date, # will force to pd.datetime.date()
    end_date = run_date, 
    per_mode = per_mode, #['Totals', 'PerGame'] 
    season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
    distance_type = 'By+Zone', # ['8ft+Range', '5ft+Range','By+Zone']
    sob = ['Base', 'Opponent'], #['Base', 'Opponent']  base= teams offense, opponent = teams defense
    database_table = 'statsteamshotzones'     
)

# PER GAME
nbaApi.request_nba_team_stats(
    season = season_str, #'YYYY-YY'
    start_date = season_start_date, # will force to pd.datetime.date()
    end_date = run_date, 
    per_mode = 'PerGame', #['Totals', 'PerGame'] 
    season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
    measure_type = ['Base', 'Advanced', 'Opponent'], #['Base', 'Advanced', 'Opponent']  
    database_table = 'statsteam'
)
# TOTALS
nbaApi.request_nba_team_stats(
    season = season_str, #'YYYY-YY'
    start_date = run_date, # will force to pd.datetime.date()
    end_date = run_date, 
    per_mode = 'Totals', #['Totals', 'PerGame'] 
    season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
    measure_type = ['Base', 'Advanced', 'Opponent'], #['Base', 'Advanced', 'Opponent']  
    database_table = 'statsteamtotals'
)

# ---------------------------------------------------------------- #
# --- PLAYER --- #
print(today, 'run date...\n')
nbaApi.request_nba_player_shotzone_data(
            season = season_str, #'YYYY-YY'
            start_date = run_date, # will force to pd.datetime.date()
            end_date = run_date, 
            per_mode = per_mode, #['Totals', 'PerGame'] 
            season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
            distance_type = 'By+Zone', # ['8ft+Range', '5ft+Range','By+Zone'],
            sob = ['Base'], #['Base', 'Opponent']  base= teams offense, opponent = teams defense
            database_table = 'statsplayershotzones'        
)
# Passing
nbaApi.request_nba_player_tracking(
        season = season_str, #'YYYY-YY'
        start_date =run_date, # will force to pd.datetime.date()
        end_date = run_date, 
        per_mode = per_mode, #['Totals', 'PerGame'] 
        season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
        measure_type = 'Passing', #['Passing','Rebounding','Drives','Possessions','Efficiency','PostTouch', 'ElbowTouch', 'PaintTouch']  
        database_table = 'statsplayerpassing'      
)
# Rebounding
nbaApi.request_nba_player_tracking(
        season = season_str, #'YYYY-YY'
        start_date =run_date, # will force to pd.datetime.date()
        end_date = run_date, 
        per_mode = per_mode, #['Totals', 'PerGame'] 
        season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
        measure_type = 'Rebounding', #['Passing','Rebounding','Drives','Possessions','Efficiency']  
        database_table = 'statsplayerrebounding'      
)

# add these measure types - 
#### 'Drives', 'Possessions', 'Efficiency' 
# maybe add these - 'PostTouch', 'ElbowTouch', 'PaintTouch'
#nbaApi.request_nba_player_tracking()

2025-11-04 run date...

nba team shot zone retrieved 18 offensive,  18 defensive loaded...
nba team stats PerGame retrieved (30, 65) loaded...
nba team stats Totals retrieved (18, 65) loaded...
2025-11-04 run date...

nba player shot zone retrieved 198 offensive loaded...
nba player Passing retrieved 198 loaded...
nba player Rebounding retrieved 185 loaded...


In [19]:
'''
These 2 playtype hits require some games before they populate on the site 
https://www.nba.com/stats/teams/playtype-post-up  check this for stats

'''

print(today, 'run date...\n')
# --- TEAM --- #
nbaApi.request_nba_team_playtype_data(
            season = season_str, #'YYYY-YY' 
            play_type = [
                'Isolation', 'Transition','PRBallHandler','PRRollman', 'Postup', 'Spotup', 
                'Handoff', 'Cut', 'OffScreen', 'OffRebound', 'Misc'
            ], 
            lid='00',
            per_mode = 'PerGame', #['Totals', 'PerGame'] 
            season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
            sob = ['offensive', 'defensive'],
            sleep_time = 2,
            database_table = 'statsteamplaytypes'
        )


print(today, 'run date...\n')
nbaApi.request_nba_player_playtype_data(
            season = season_str, #'YYYY-YY' 
            play_type = [
                'Isolation', 'Transition','PRBallHandler','PRRollman', 'Postup', 'Spotup', 
                'Handoff', 'Cut', 'OffScreen', 'OffRebound', 'Misc'
            ], 
            lid='00',
            per_mode = 'PerGame', #['Totals', 'PerGame']
            season_type = season_type, #['Regular+Season', 'PlayIn', 'Playoffs']
            sob = ['offensive'], #'offensive','defensive'],
            sleep_time = 2,
            database_table = 'statsplayerplaytypes'
)


2025-11-04 run date...

retrieved nba team play types: 11 , sob: 2 , teams: 30
2025-11-04 run date...

retrieved nba player play types: 11 , sob: 1 , players: 273


# scratch

In [43]:
from nba_api.stats.endpoints import leaguegamefinder 
# Fetch all games for the current season
seasons = '2025-26'
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=seasons, league_id_nullable='00')
g = gamefinder.get_data_frames()[0]
g

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612750,MIN,Minnesota Timberwolves,0022500008,2025-10-27,MIN vs. DEN,L,240,114,...,0.828,7,26,33,19,8,9,14,22,-13.0
1,22025,1610612760,OKC,Oklahoma City Thunder,0022500119,2025-10-27,OKC @ DAL,W,239,101,...,0.846,9,46,55,21,5,5,13,23,7.0
2,22025,1610612753,ORL,Orlando Magic,0022500114,2025-10-27,ORL @ PHI,L,240,124,...,0.684,13,30,43,27,4,7,13,22,-12.0
3,22025,1610612759,SAS,San Antonio Spurs,0022500118,2025-10-27,SAS vs. TOR,W,239,121,...,0.833,7,37,44,29,11,3,20,17,14.0
4,22025,1610612739,CLE,Cleveland Cavaliers,0022500007,2025-10-27,CLE @ DET,W,240,116,...,0.771,12,36,48,25,11,4,19,25,21.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246,12025,1610612756,PHX,Phoenix Suns,0012500001,2025-10-03,PHX @ LAL,W,241,103,...,0.538,10,40,50,29,9,5,16,31,22.0
247,12025,1610612747,LAL,Los Angeles Lakers,0012500001,2025-10-03,LAL vs. PHX,L,240,81,...,0.784,11,35,46,10,9,7,22,25,-22.0
248,12025,15016,MEL,Melbourne United,0012500009,2025-10-03,MEL @ NOP,L,240,97,...,0.650,13,35,48,23,7,1,17,18,-10.2
249,12025,1610612752,NYK,New York Knicks,0012500008,2025-10-02,PHI @ NYK,W,238,99,...,0.656,20,38,58,19,13,2,17,26,15.0
